# Tutorial: Multi-Agent Taskboard Demo

This notebook keeps the worktree demo intentionally small:

- import a shared `model` and base `agent` from `model_setup.py`
- copy `taskboard_demo` into a disposable Git repository
- create two isolated workspaces with Git worktrees
- run two inline pyflow requests concurrently with `await asyncio.gather(...)`
- inspect the resulting branch state in each workspace


## Notebook Setup

The notebook now lives under `demo/`, so the first cell resolves the repository root and imports the shared runtime setup from `model_setup.py` plus the low-signal helper utilities from `demo.utils`.


In [ ]:
import asyncio
import sys

from pathlib import Path

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from demo.multi_agent_utils import format_name_status_diff, git, prepare_taskboard_demo, run_command
from model_setup import agent, model
from pyflow import code, docs, tests

REPO_ROOT, model, agent


/Users/goldenberg/Developer/pyflow/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
+----------------------------------------------------------------------+
|  OpenHands SDK v1.14.0                                               |
|                                                                      |
|  Report a bug: github.com/OpenHands/software-agent-sdk/issues        |
|  Get help: openhands.dev/joinslack                                   |
|  Scale up: openhands.dev/product/sdk                                 |
|                                                                      |
|  Set OPENHANDS_SUPPRESS_BANNER=1 to hide this message                |
+----------------------------------------------------------------------+



[03/24/26 03:42:07] INFO     Using credentials directory: /Users/goldenberg/.openhands/auth       ]8;id=836400;file:///Users/goldenberg/Developer/pyflow/.venv/lib/python3.12/site-packages/openhands/sdk/llm/auth/credentials.py\credentials.py]8;;\:]8;id=315037;file:///Users/goldenberg/Developer/pyflow/.venv/lib/python3.12/site-packages/openhands/sdk/llm/auth/credentials.py#57\57]8;;\

[03/24/26 03:42:07] INFO     Using existing OpenAI credentials                                        ]8;id=372593;file:///Users/goldenberg/Developer/pyflow/.venv/lib/python3.12/site-packages/openhands/sdk/llm/auth/openai.py\openai.py]8;;\:]8;id=42836;file:///Users/goldenberg/Developer/pyflow/.venv/lib/python3.12/site-packages/openhands/sdk/llm/auth/openai.py#659\659]8;;\

(PosixPath('/Users/goldenberg/Developer/pyflow-clones/pyflow2'),
 AIModel(llm=LLM(model='openai/gpt-5.2-codex', api_key=SecretStr('**********'), base_url='https://chatgpt.com/backend-api/codex', api_version=None, aws_access_key_id=None, aws_secret_access_key=None, aws_region_name=None, openrouter_site_url='https://docs.all-hands.dev/', openrouter_app_name='OpenHands', num_retries=5, retry_multiplier=8.0, retry_min_wait=8, retry_max_wait=64, timeout=300, max_message_chars=30000, temperature=None, top_p=None, top_k=None, max_input_tokens=272000, max_output_tokens=None, model_canonical_name=None, extra_headers={'originator': 'codex_cli_rs', 'OpenAI-Beta': 'responses=experimental', 'User-Agent': 'openhands-sdk (Darwin; x86_64)', 'chatgpt-account-id': '774a6b29-2e05-40fd-8722-a966d2aff14b'}, input_cost_per_token=None, output_cost_per_token=None, ollama_base_url=None, stream=True, drop_params=True, modify_params=True, disable_vision=None, disable_stop_word=False, caching_prompt=True, log_com

## Materialize A Disposable Demo Repo

The base project stays unchanged. This cell uses `demo.utils.prepare_taskboard_demo(...)` to copy `taskboard_demo` into a temporary directory, create a fresh Git repository, and prepare one worktree per feature branch.


In [2]:
demo = prepare_taskboard_demo(
    repo_root=REPO_ROOT,
    run_id="multi-agent-demo",
)
demo_repo = demo.repo
priority_branch = demo.priority_branch
due_date_branch = demo.due_date_branch
priority_worktree = demo.priority_worktree
due_date_worktree = demo.due_date_worktree

print(f"demo_repo_path={demo.repo_path}")
print(f"priority_worktree={priority_worktree.path}")
print(f"due_date_worktree={due_date_worktree.path}")
print(run_command(demo.repo_path, sys.executable, "-m", "pytest", "tests"))


demo_repo_path=/private/var/folders/yl/1wx4p1fd0hl654v2jk7cxygc0000gn/T/pyflow-taskboard-nb-fp3wemxx/taskboard-demo
priority_worktree=/private/var/folders/yl/1wx4p1fd0hl654v2jk7cxygc0000gn/T/pyflow-taskboard-nb-fp3wemxx/taskboard-demo-worktrees/multi-agent-demo/priority
due_date_worktree=/private/var/folders/yl/1wx4p1fd0hl654v2jk7cxygc0000gn/T/pyflow-taskboard-nb-fp3wemxx/taskboard-demo-worktrees/multi-agent-demo/due-date
$ /Users/goldenberg/Developer/pyflow/.venv/bin/python -m pytest tests
returncode=0
stdout:
============================= test session starts ==============================
platform darwin -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0
rootdir: /private/var/folders/yl/1wx4p1fd0hl654v2jk7cxygc0000gn/T/pyflow-taskboard-nb-fp3wemxx/taskboard-demo
configfile: pyproject.toml
plugins: anyio-4.12.1, libtmux-0.55.0, asyncio-1.3.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 4 items

tests/test

## Run Two Inline Requests Concurrently

The only runtime setup comes from `model_setup.py`. Two workspace-specific agents are derived from the imported base `agent`, and the requests stay inline so the DSL remains visible in one place.


In [3]:
priority_agent = agent.replacing(workspace=priority_worktree.path)
due_date_agent = agent.replacing(workspace=due_date_worktree.path)

assert priority_agent.model is model
assert due_date_agent.model is model

priority_session, due_date_session = await asyncio.gather(
    priority_agent.run_async(
        """
        Implement priority support for this taskboard app.

        Requirements:
        - Add a `priority` field with allowed values `low`, `medium`, and `high`.
        - Extend `taskboard add` with `--priority`.
        - Persist priority in JSON storage.
        - Show priority in `taskboard list`.
        - Sort incomplete tasks by priority before creation time and id.
        - Update `README.md` and `tests/test_cli.py`.

        Constraints:
        - Work only in the current branch and workspace.
        - Keep backward compatibility for existing JSON data.
        - Stage the final changes and create a commit with the message `Add priority support`.
        """.strip()
        @ docs("README.md")
        @ code("taskboard/models.py")
        @ code("taskboard/storage.py")
        @ code("taskboard/cli.py")
        @ code("tests/test_cli.py")
        >> tests("python -m pytest tests")
    ),
    due_date_agent.run_async(
        """
        Implement due-date support for this taskboard app.

        Requirements:
        - Add an optional `due_date` field stored as `YYYY-MM-DD`.
        - Extend `taskboard add` with `--due`.
        - Persist due dates in JSON storage.
        - Show due dates in `taskboard list`.
        - Add `taskboard list --overdue` for incomplete overdue tasks.
        - Sort incomplete tasks by due date before creation time and id.
        - Update `README.md` and `tests/test_cli.py`.

        Constraints:
        - Work only in the current branch and workspace.
        - Keep backward compatibility for existing JSON data.
        - Stage the final changes and create a commit with the message `Add due date support`.
        """.strip()
        @ docs("README.md")
        @ code("taskboard/models.py")
        @ code("taskboard/storage.py")
        @ code("taskboard/cli.py")
        @ code("tests/test_cli.py")
        >> tests("python -m pytest tests")
    ),
)

ConversationRunError: Conversation run failed for id=827ec661-a478-43a5-83f3-4396183d520c: [Errno 9] Bad file descriptor

In [ ]:
priority_session

In [ ]:
due_date_session

## Inspect The Resulting Branches

Each agent worked against its own Git worktree. This final cell shows the latest commit, reruns the tests in each workspace, and prints the branch diff relative to `main`.


In [ ]:
for label, branch, worktree in (
    ("priority", priority_branch, priority_worktree),
    ("due-date", due_date_branch, due_date_worktree),
):
    print(f"\n[{label}] workspace={worktree.path}")
    print(git(worktree.path, "status", "--short", "--branch").stdout.strip())
    print(git(worktree.path, "log", "--oneline", "--max-count=1").stdout.strip())
    print(run_command(worktree.path, sys.executable, "-m", "pytest", "tests"))
    diff = demo_repo.diff(
        left="main",
        right=branch,
        merge_base=True,
        mode="name_status",
    )
    print(format_name_status_diff(diff.name_status_entries))


## Takeaway

`Agent.parallel(...)` still fits shared-workspace fanout. When each worker needs its own branch or worktree, deriving two `Agent` values with different `workspace` paths and awaiting `asyncio.gather(...)` keeps the notebook flow direct while still showing the core pyflow DSL.
